# Task 2: Change Point Modeling and Insight Generation
### Brent Oil Price Change Point Analysis — Birhan Energies

**Objective:** Apply Bayesian change point detection to Brent oil log
returns to identify and quantify structural breaks, then associate
detected change points with researched real-world events.

## 1. Data Preparation

In [1]:
import sys
sys.path.append("../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
os.environ["PYTENSOR_FLAGS"] = "cxx=,mode=NUMBA"

# Your PyMC / PyTensor code goes below this line
import pymc as pm
import arviz as az

from src.data_prep import load_price_data, add_log_returns, load_events
from src.change_point_model import build_change_point_model, sample_model
from src.event_matching import match_change_point_to_events

plt.rcParams["figure.figsize"] = (12, 5)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


In [4]:
prices = load_price_data("../data/raw/BrentOilPrices.csv")
prices = add_log_returns(prices)
events = load_events("../data/raw/majorEvents.csv")

log_returns_series = prices["log_return"].dropna()
log_returns = log_returns_series.values

print(f"Log returns: {len(log_returns)} observations")
print(f"Date range: {log_returns_series.index.min().date()} to {log_returns_series.index.max().date()}")

Log returns: 9010 observations
Date range: 1987-05-21 to 2022-11-14


In [5]:
# Resample daily prices to weekly (Friday close) to make MCMC sampling
# computationally tractable without a working C++ compiler.
weekly_prices = prices["Price"].resample("W").last().dropna()
weekly_log_returns_series = np.log(weekly_prices) - np.log(weekly_prices.shift(1))
weekly_log_returns_series = weekly_log_returns_series.dropna()

weekly_log_returns = weekly_log_returns_series.values

print(f"Weekly observations: {len(weekly_log_returns)}")
print(f"Date range: {weekly_log_returns_series.index.min().date()} to {weekly_log_returns_series.index.max().date()}")

Weekly observations: 1852
Date range: 1987-05-31 to 2022-11-20


In [ ]:
import time

start = time.time()

model = build_change_point_model(weekly_log_returns)
trace = sample_model(model, draws=2000, tune=1000, chains=4)

elapsed = time.time() - start
print(f"Elapsed: {elapsed:.1f} seconds")

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>Metropolis: [tau]
>NUTS: [mu1, mu2, sigma]


c:\projects\change-point-analysis-brent-oil\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

## 2. Build and Sample the Bayesian Change Point Model

Model structure:
- `tau` — discrete uniform prior over all possible days (the unknown switch point)
- `mu1`, `mu2` — the mean log return before/after tau
- `sigma` — shared standard deviation across both regimes
- `obs` — Normal likelihood connecting the model to the observed log returns

**Note:** this may take several minutes to sample given ~9,000 daily
observations spanning 35 years.

In [6]:
model = build_change_point_model(log_returns)

with model:
    trace_test = pm.sample(
        draws=200,
        tune=200,
        chains=2,
        return_inferencedata=True,
        target_accept=0.8,
    )

c:\projects\change-point-analysis-brent-oil\.venv\Lib\site-packages\pytensor\link\c\cmodule.py:2100: UserWarning: `pytensor.config.cxx` is not an identifiable `g++` compiler. PyTensor will disable compiler optimizations specific to `g++`. At worst, this could cause slow downs.
Those parameters can be added manually via the `cxxflags` setting.
  warnings.warn(



You can find the C code in this temporary file: C:\Users\LIYADA~1\AppData\Local\Temp\pytensor_compilation_error_o7mx08qd


CompileError: Compilation failed (return status=2):
cl.exe -shared -g -Wno-c++11-narrowing -fno-exceptions -fno-unwind-tables -fno-asynchronous-unwind-tables -DNPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION -m64 -DMS_WIN64 -I"c:\projects\change-point-analysis-brent-oil\.venv\Lib\site-packages\numpy\_core\include" -I"C:\Program Files\Python311\include" -I"c:\projects\change-point-analysis-brent-oil\.venv\Lib\site-packages\pytensor\link\c\c_code" -L"C:\Program Files\Python311\libs" -L"C:\Program Files\Python311" -o "C:\Users\Liya Daniel\AppData\Local\PyTensor\compiledir_Windows-10-10.0.21996-SP0-Intel64_Family_6_Model_140_Stepping_1_GenuineIntel-3.11.0-64\lazylinker_ext\lazylinker_ext.pyd" "C:\Users\Liya Daniel\AppData\Local\PyTensor\compiledir_Windows-10-10.0.21996-SP0-Intel64_Family_6_Model_140_Stepping_1_GenuineIntel-3.11.0-64\lazylinker_ext\mod.cpp" "C:\Program Files\Python311\python311.dll"
Microsoft (R) C/C++ Optimizing Compiler Version 19.51.36248 for x64
Copyright (C) Microsoft Corpo

In [3]:
model = build_change_point_model(log_returns)
trace = sample_model(model, draws=2000, tune=1000, chains=4)

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>Metropolis: [tau]
>NUTS: [mu1, mu2, sigma]


c:\projects\change-point-analysis-brent-oil\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

ValueError: Not enough samples to build a trace.

## 3. Check Convergence

We check R-hat (should be close to 1.0) and visually inspect trace
plots for the "fuzzy caterpillar" pattern — overlapping, noisy bands
with no drift or separation between chains.

In [ ]:
summary = az.summary(trace, var_names=["tau", "mu1", "mu2", "sigma"])
summary

In [ ]:
az.plot_trace(trace, var_names=["tau", "mu1", "mu2", "sigma"])
plt.tight_layout()
plt.show()

If any R-hat values are above ~1.01, the sampler likely needs more
iterations, more tuning, or a reparameterization — do not proceed to
interpretation until convergence looks solid.

## 4. Identify the Change Point

We convert tau's posterior (a day index) back into an actual calendar
date, and examine how sharp/certain the posterior peak is.

In [ ]:
az.plot_posterior(trace, var_names=["tau"])
plt.title("Posterior Distribution of the Change Point (tau)")
plt.show()

In [ ]:
tau_samples = trace.posterior["tau"].values.flatten()
tau_mode = int(pd.Series(tau_samples).mode()[0])
tau_median = int(np.median(tau_samples))

change_date_mode = log_returns_series.index[tau_mode]
change_date_median = log_returns_series.index[tau_median]

print(f"Most probable change point (mode): Day {tau_mode} -> {change_date_mode.date()}")
print(f"Median change point:               Day {tau_median} -> {change_date_median.date()}")
print(f"Tau posterior std (days):          {tau_samples.std():.1f}")

A small posterior standard deviation (few days/weeks) indicates high
certainty about *when* the break occurred. A large spread indicates
the model is unsure, and any single point estimate should be treated
cautiously.

## 5. Quantify the Impact

We compare the posterior distributions of mu1 and mu2 to make a
probabilistic, quantified statement about the size of the shift.

In [ ]:
az.plot_posterior(trace, var_names=["mu1", "mu2"])
plt.show()

In [ ]:
mu1_samples = trace.posterior["mu1"].values.flatten()
mu2_samples = trace.posterior["mu2"].values.flatten()

mu1_mean = mu1_samples.mean()
mu2_mean = mu2_samples.mean()

# Probability that mu2 > mu1 (i.e. the shift was an increase)
prob_increase = (mu2_samples > mu1_samples).mean()

# Convert mean log return to an approximate annualized/aggregate price effect
pct_daily_change = (np.exp(mu2_mean) - np.exp(mu1_mean)) / np.exp(mu1_mean) * 100

print(f"Mean daily log return BEFORE change point: {mu1_mean:.5f}")
print(f"Mean daily log return AFTER change point:  {mu2_mean:.5f}")
print(f"P(mu2 > mu1) [probability the shift was an increase]: {prob_increase:.3f}")
print(f"Approx. relative change in typical daily return: {pct_daily_change:.2f}%")

## 6. Associate the Change Point with Researched Events

We search the compiled events dataset for anything within a 90-day
window of the detected change point. **This identifies correlation in
time only** — see Task 1's discussion on why this does not, by itself,
establish causation.

In [ ]:
matched_events = match_change_point_to_events(
    change_date=change_date_median,
    events_df=events,
    window_days=90,
)
matched_events

## 7. Written Insight

*(Fill in once the above cells have been run with real output — use
this as a template.)*

> The Bayesian change point model detected a structural break around
> **[change_date_median]**, with a tau posterior standard deviation of
> **[X] days**, indicating [high/moderate/low] certainty about the exact
> timing. The mean daily log return shifted from **[mu1_mean]** to
> **[mu2_mean]**, with a posterior probability of **[prob_increase]**
> that this represents a genuine increase. The nearest researched event
> within the search window was **[matched event, if any]**, occurring
> **[N] days** from the detected change point — a plausible but not
> confirmed trigger for this shift, given the correlation-vs-causation
> caveats discussed in Task 1.